# Import and Setup

In [13]:
import os
import json
import ast
from openai import OpenAI

openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key
client = OpenAI()
MODEL = "gpt-5.4"

# LLM Functions

In [14]:
def generate_data_science_tasks(n: int = 10) -> list:
    prompt = f"""Brainstorm a list of {n} descriptions of AI tasks that can be evaluated using a ML model and HuggingFace datasets.
Restrict to tasks with datasets that have less than a million samples.
Each description should specify both the task type and the dataset,
Return ONLY a valid Python list of strings, no explanation."""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    raw = response.choices[0].message.content.strip()
    return ast.literal_eval(raw)


tasks = generate_data_science_tasks(10)
tasks

['Sentiment classification on the IMDb movie reviews dataset',
 'Natural language inference on the SNLI dataset',
 'Question answering on the SQuAD v1.1 dataset',
 'Summarization on the XSum dataset',
 'Toxic comment classification on the Jigsaw Toxicity dataset',
 'Named entity recognition on the CoNLL-2003 dataset',
 'Paraphrase identification on the MRPC dataset',
 'Commonsense reasoning multiple-choice QA on the HellaSwag dataset',
 'Code generation or code completion on the HumanEval dataset',
 'Machine translation from English to German on the IWSLT2017 dataset']

In [ ]:
def generate_notebook(task: str, notebook_dir: str) -> str:
    # Step 0: Name the notebook
    name_response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": f"""Generate a short, descriptive filename for a Jupyter notebook about this AI task:

Task: {task}

Requirements:
- Use snake_case
- End with .ipynb
- Be concise (3-6 words)
- Return ONLY the filename, nothing else."""}],
    )
    notebook_name = name_response.choices[0].message.content.strip()
    if not notebook_name.endswith(".ipynb"):
        notebook_name += ".ipynb"

    # Step 1: Plan
    plan_response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": f"""You are an expert data scientist. Plan a Jupyter notebook workflow for this AI task:

Task: {task}

Use HuggingFace datasets and simple, cheap, locally-runnable models (e.g. sentence-transformers, TF-IDF + sklearn, small HuggingFace models via transformers.pipeline).
Write a concise step-by-step plan for the notebook."""}],
    )
    plan = plan_response.choices[0].message.content.strip()

    # Step 2: Generate notebook JSON
    nb_response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": f"""You are an expert data scientist. Generate a complete Jupyter notebook as valid JSON for this AI task.

Task: {task}

Plan:
{plan}

Requirements:
- Use HuggingFace datasets to load data
- Use simple, cheap, locally-runnable models for inference (e.g. sentence-transformers, TF-IDF + sklearn, small HuggingFace models via transformers.pipeline)
- Include cells for imports, data loading, inference, and evaluation
- Only use one model and one dataset/subset of the dataset
- Only use code cells (no markdown cells)
- No plots or visualizations
- The notebook must be valid .ipynb JSON (nbformat 4)
- Return ONLY the raw JSON, no markdown fences or explanation."""}],
    )
    raw = nb_response.choices[0].message.content.strip()
    if raw.startswith("```"):
        raw = raw.split("```", 2)[1]
        if raw.startswith("json"):
            raw = raw[4:]
        raw = raw.rsplit("```", 1)[0].strip()

    nb = json.loads(raw)
    os.makedirs(notebook_dir, exist_ok=True)
    path = os.path.join(notebook_dir, notebook_name)
    with open(path, "w") as f:
        json.dump(nb, f, indent=1)
    print(f"Notebook saved to {path}")
    return path

In [ ]:
def generate_n_variations(notebook_path: str, n: int = 3) -> list:
    with open(notebook_path, "r") as f:
        original_nb = json.load(f)

    cells_text = []
    for cell in original_nb.get("cells", []):
        source = "".join(cell.get("source", []))
        if source.strip():
            cells_text.append(source)
    original_content = "\n\n---\n\n".join(cells_text)

    original_name = os.path.splitext(os.path.basename(notebook_path))[0]
    notebook_dir = os.path.dirname(notebook_path)
    variations_dir = os.path.join(notebook_dir, "variations")
    os.makedirs(variations_dir, exist_ok=True)

    # Step 0: Plan all variations upfront, returning a name -> description dict
    plan_response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": f"""You are an expert data scientist. Plan {n} variations of the following Jupyter notebook.

Original notebook:
{original_content}

Return ONLY a valid Python dictionary mapping a snake_case name to a concise description for each variation, no explanation. The name will be the file name for the notebook without extensions."""}],
    )
    raw_plans = plan_response.choices[0].message.content.strip()
    variation_plans = ast.literal_eval(raw_plans)

    paths = []
    for name, plan in variation_plans.items():
        # Step 1: Generate the variation notebook according to its plan
        nb_response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": f"""You are an expert data scientist. Generate a Jupyter notebook as valid JSON implementing this variation of an existing notebook.

Original notebook:
{original_content}

Variation plan:
{plan}

Requirements:
- Use HuggingFace datasets to load data
- Use simple, cheap, locally-runnable models for inference (e.g. sentence-transformers, TF-IDF + sklearn, small HuggingFace models via transformers.pipeline)
- No OpenAI API calls
- Include cells for imports, data loading, inference, and evaluation
- Only use one model and one dataset/subset of the dataset
- Only use code cells (no markdown cells)
- No plots or visualizations
- The notebook must be valid .ipynb JSON (nbformat 4)
- Return ONLY the raw JSON, no markdown fences or explanation."""}],
        )
        raw = nb_response.choices[0].message.content.strip()
        if raw.startswith("```"):
            raw = raw.split("```", 2)[1]
            if raw.startswith("json"):
                raw = raw[4:]
            raw = raw.rsplit("```", 1)[0].strip()

        nb = json.loads(raw)
        variation_name = f"{name}.ipynb"
        path = os.path.join(variations_dir, variation_name)
        with open(path, "w") as f:
            json.dump(nb, f, indent=1)
        print(f"Variation saved to {path}")
        paths.append(path)

    return paths

In [17]:
def generate_synthetic_notebook(notebook_path: str) -> str:
    with open(notebook_path, "r") as f:
        nb = json.load(f)

    new_cells = []
    for cell in nb.get("cells", []):
        if cell.get("cell_type") != "code":
            new_cells.append(cell)
            continue

        source = "".join(cell.get("source", []))

        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": f"""You are an expert data scientist. Given this Jupyter notebook cell, create a synthetic version that avoids per-row API calls.

Original cell code:
{source}

Requirements:
- If this cell makes per-row API calls (e.g., calling OpenAI in a loop over dataset rows), replace them with synthetic/fake outputs
- You may make a single model call upfront to generate a batch of realistic synthetic examples if needed for realism
- Preserve all original variable names and the overall workflow structure
- Format the output as: first the original code wrapped in a triple-quoted string assigned to `original_code`, then the modified synthetic code below it in the same cell
- If the cell has NO per-row API calls, return the original code unchanged (no wrapping needed)
- Return ONLY the Python code, no explanation or markdown fences."""}],
        )
        new_source = response.choices[0].message.content.strip()
        if new_source.startswith("```"):
            new_source = new_source.split("```", 2)[1]
            if new_source.startswith("python"):
                new_source = new_source[6:]
            new_source = new_source.rsplit("```", 1)[0].strip()

        new_cell = dict(cell)
        new_cell["source"] = new_source
        new_cell["outputs"] = []
        new_cells.append(new_cell)

    nb["cells"] = new_cells
    base, ext = os.path.splitext(notebook_path)
    synthetic_path = base + "_synthetic" + ext
    with open(synthetic_path, "w") as f:
        json.dump(nb, f, indent=1)
    print(f"Synthetic notebook saved to {synthetic_path}")
    return synthetic_path

# Initial Dataset

- Select best suited task from list
- Create initial notebook and store in "../notebooks/batch_2" folder.
- Create 5 variations of the initial notebook.
- For each variation, create 2-4 new variations of the variation.

In [18]:
selected_task = "Emotion classification on the Emotion dataset from HuggingFace Datasets."

In [19]:
NOTEBOOK_DIR = "../notebooks/batch_2"
notebook_path = generate_notebook(selected_task, NOTEBOOK_DIR)

Notebook saved to ../notebooks/batch_2/emotion_classification_emotion_dataset.ipynb


In [ ]:
variation_paths = generate_n_variations(notebook_path, n=5)

In [ ]:
import random

for vp in variation_paths:
    n_sub = random.randint(2, 4)
    generate_n_variations(vp, n=n_sub)